## Setup: Imports and Drive Connection


In [ ]:
# here we are importing all the required libraries
import tensorflow as tf
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense, Dropout, RandomFlip, RandomRotation, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
import matplotlib.pyplot as plt
import os
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
import cv2

# here we are connecting to google drive
print("connecting to google drive...")
from google.colab import drive
drive.mount('/content/drive')
print("drive mounted successfully")


Connecting to Google Drive...
Mounted at /content/drive
Drive mounted successfully.


## Unzip Data and Define Variables


In [ ]:
# here we are unzipping the dataset from google drive
print("unzipping the data zip file from drive...")
ZIP_PATH = "/content/drive/MyDrive/DATA.zip"
!unzip -o -q {ZIP_PATH} -d "/content/"
print("data is unzipped and ready in content")

# here we are defining the main project variables
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 4
EPOCHS_STAGE_1 = 15
EPOCHS_STAGE_2 = 10

# here we are setting the paths to the training validation and testing folders
TRAIN_DIR = "/content/DATA/Training(70%)"
VALID_DIR = "/content/DATA/Validation(20%)"
TEST_DIR = "/content/DATA/Testing(10%)"

# here we are defining the learning rates we want to test
LEARNING_RATES_TO_TEST = [1e-5, 5e-5, 1e-4]


Unzipping the DATA.zip file from Drive...
Data is unzipped and ready in /content/.


##Define Preprocessing and Data Functions


In [ ]:
# here we are defining the cv2 preprocessing function
def apply_preprocessing(image_array):
    # here we are converting the image to uint8 for opencv
    image_uint8 = image_array.astype(np.uint8)
    # here we are applying median blur to remove noise
    image_blur = cv2.medianBlur(image_uint8, 5)
    # here we are converting the image to lab color space for better contrast adjustment
    image_lab = cv2.cvtColor(image_blur, cv2.COLOR_RGB2LAB)
    l_channel, a_channel, b_channel = cv2.split(image_lab)
    # here we are applying clahe to the l channel for contrast enhancement
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l_channel)
    # here we are merging the channels and converting back to rgb
    merged = cv2.merge((cl, a_channel, b_channel))
    final_image = cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)
    # here we are returning the processed image as float32
    return final_image.astype(np.float32)

# here we are defining the tensorflow wrapper for the preprocessing function
@tf.function
def tf_preprocess_wrapper(image, label):
    [image,] = tf.numpy_function(apply_preprocessing, [image], [tf.float32])
    image.set_shape([IMAGE_SIZE[0], IMAGE_SIZE[1], 3])
    return image, label

# here we are creating a function to prepare and preprocess all datasets
def create_datasets():
    print("creating and preprocessing datasets...")
    train_ds = tf.keras.utils.image_dataset_from_directory(
        TRAIN_DIR, label_mode="categorical", seed=123,
        image_size=IMAGE_SIZE, batch_size=BATCH_SIZE
    )

    val_ds = tf.keras.utils.image_dataset_from_directory(
        VALID_DIR, label_mode="categorical", seed=123,
        image_size=IMAGE_SIZE, batch_size=BATCH_SIZE
    )

    test_ds = tf.keras.utils.image_dataset_from_directory(
        TEST_DIR, label_mode="categorical", image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE, shuffle=False
    )

    # here we are calculating class weights to handle imbalance
    print("calculating class weights...")
    train_labels = np.concatenate([np.argmax(y.numpy(), axis=1) for x, y in train_ds])
    class_weights = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
    class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}
    print(f"calculated weights: {class_weight_dict}")

    # here we are applying preprocessing and optimizing with cache and prefetch
    AUTOTUNE = tf.data.AUTOTUNE
    train_ds = train_ds.unbatch().map(tf_preprocess_wrapper, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(buffer_size=AUTOTUNE)
    val_ds = val_ds.unbatch().map(tf_preprocess_wrapper, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(buffer_size=AUTOTUNE)
    test_ds = test_ds.unbatch().map(tf_preprocess_wrapper, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(buffer_size=AUTOTUNE)

    print("datasets are ready")
    return train_ds, val_ds, test_ds, class_weight_dict


#Define Model Building Function


In [ ]:
# here we are defining a function to build the resnet model
def build_model():
    print("building new resnet50 model...")
    # here we are loading the resnet50 base model with imagenet weights
    base_model = ResNet50(weights='imagenet', include_top=False,
                          input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3))
    base_model.trainable = False

    # here we are creating the model structure with augmentation and preprocessing
    inputs = Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3))
    x = RandomFlip('horizontal')(inputs)
    x = RandomRotation(0.1)(x)
    x = Lambda(preprocess_input)(x)
    x = base_model(x, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.3)(x)
    outputs = Dense(NUM_CLASSES, activation='softmax')(x)
    model = Model(inputs, outputs)

    # here we are returning both the model and the base model
    return model, base_model


## Run the Hyperparameter Tuning Loop


In [ ]:
# here we are running the tuning loop for different learning rates

# here we are creating a dictionary to store all final results
results_log = {}

# here we are looping through each learning rate to test
for lr in LEARNING_RATES_TO_TEST:
    print(f"starting test for learning rate {lr}")

    # here we are creating fresh datasets and model for each run
    train_dataset, validation_dataset, test_dataset, class_weight_dict = create_datasets()
    model, base_model = build_model()

    # here we are running stage 1 head training
    print("running stage 1 head training")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    history = model.fit(
        train_dataset,
        validation_data=validation_dataset,
        epochs=EPOCHS_STAGE_1,
        class_weight=class_weight_dict,
        verbose=0
    )
    print("stage 1 complete")

    # here we are running stage 2 fine tuning
    print(f"running stage 2 fine tuning with learning rate {lr}")
    base_model.trainable = True
    for layer in base_model.layers[:-30]:
        layer.trainable = False

    # here we are recompiling the model with the new learning rate
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy',
                 tf.keras.metrics.Precision(name='precision'),
                 tf.keras.metrics.Recall(name='recall')]
    )

    # here we are training the model for the fine tuning stage
    history_finetune = model.fit(
        train_dataset,
        validation_data=validation_dataset,
        epochs=EPOCHS_STAGE_2,
        initial_epoch=history.epoch[-1],
        class_weight=class_weight_dict,
        verbose=0
    )
    print("stage 2 complete")

    # here we are evaluating the final model performance
    print("evaluating final model")
    results = model.evaluate(test_dataset, verbose=0)

    # here we are storing the evaluation metrics
    metrics = {}
    metrics['loss'] = results[0]
    metrics['accuracy'] = results[1]
    metrics['precision'] = results[2]
    metrics['recall'] = results[3]
    if (metrics['precision'] + metrics['recall']) > 0:
        metrics['f1_score'] = 2 * (metrics['precision'] * metrics['recall']) / (metrics['precision'] + metrics['recall'])
    else:
        metrics['f1_score'] = 0.0

    # here we are saving the metrics for this learning rate
    results_log[f"LR_{lr}"] = metrics

    print(f"results for learning rate {lr}")
    print(f"accuracy {metrics['accuracy']:.4f}, f1 score {metrics['f1_score']:.4f}")

    # here we are saving the model for this learning rate
    model_name = f"resnet_tuned_lr_{lr}.h5"
    print(f"saving model to {model_name}")
    model.save(f"/content/drive/MyDrive/MODELS/{model_name}")

# here we are confirming that all tuning runs are complete
print("hyperparameter tuning complete")



--- STARTING TEST FOR LEARNING RATE: 1e-05 ---
Creating and preprocessing datasets...
Found 2297 files belonging to 4 classes.
Found 573 files belonging to 4 classes.
Found 394 files belonging to 4 classes.
Calculating class weights...
Calculated weights: {0: np.float64(0.8687594553706506), 1: np.float64(0.8727203647416414), 2: np.float64(1.817246835443038), 3: np.float64(0.8674471299093656)}
Datasets are ready.
Building new ResNet50 model...
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
--- Running Stage 1 (Head Training) ---


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


Stage 1 complete.
--- Running Stage 2 (Fine-Tuning) with LR: 1e-05 ---
Stage 2 complete.
--- Evaluating final model ---


--- RESULTS FOR LR 1e-05 ---
Accuracy: 0.5254, F1-Score: 0.5292
Saving model to resnet_tuned_lr_1e-05.h5

--- STARTING TEST FOR LEARNING RATE: 5e-05 ---
Creating and preprocessing datasets...
Found 2297 files belonging to 4 classes.
Found 573 files belonging to 4 classes.
Found 394 files belonging to 4 classes.
Calculating class weights...
Calculated weights: {0: np.float64(0.8687594553706506), 1: np.float64(0.8727203647416414), 2: np.float64(1.817246835443038), 3: np.float64(0.8674471299093656)}
Datasets are ready.
Building new ResNet50 model...
--- Running Stage 1 (Head Training) ---


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


Stage 1 complete.
--- Running Stage 2 (Fine-Tuning) with LR: 5e-05 ---
Stage 2 complete.
--- Evaluating final model ---


--- RESULTS FOR LR 5e-05 ---
Accuracy: 0.5609, F1-Score: 0.5459
Saving model to resnet_tuned_lr_5e-05.h5

--- STARTING TEST FOR LEARNING RATE: 0.0001 ---
Creating and preprocessing datasets...
Found 2297 files belonging to 4 classes.
Found 573 files belonging to 4 classes.
Found 394 files belonging to 4 classes.
Calculating class weights...
Calculated weights: {0: np.float64(0.8687594553706506), 1: np.float64(0.8727203647416414), 2: np.float64(1.817246835443038), 3: np.float64(0.8674471299093656)}
Datasets are ready.
Building new ResNet50 model...
--- Running Stage 1 (Head Training) ---


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


Stage 1 complete.
--- Running Stage 2 (Fine-Tuning) with LR: 0.0001 ---
Stage 2 complete.
--- Evaluating final model ---


--- RESULTS FOR LR 0.0001 ---
Accuracy: 0.5228, F1-Score: 0.5115
Saving model to resnet_tuned_lr_0.0001.h5

--- HYPERPARAMETER TUNING COMPLETE ---


## Print Final Results Table


In [ ]:
# here we are printing the final tuning results summary
print("final tuning results summary")
print("lr\t\t| accuracy\t| f1 score\t| precision\t| recall\t| loss")
print("-----------------------------------------------------------------------------------------------")

# here we are setting up variables to track the best learning rate
best_lr = None
best_accuracy = 0.0

# here we are looping through the results to display each learning rate performance
for lr, metrics in results_log.items():
    print(f"{lr}\t| {metrics['accuracy']:.4f}\t\t| {metrics['f1_score']:.4f}\t\t| {metrics['precision']:.4f}\t\t| {metrics['recall']:.4f}\t\t| {metrics['loss']:.4f}")
    if metrics['accuracy'] > best_accuracy:
        best_accuracy = metrics['accuracy']
        best_lr = lr

# here we are printing the final conclusion
print("conclusion")
print(f"tuning complete the best learning rate was {best_lr} with {best_accuracy:.4f} accuracy")


--- Final Tuning Results Summary ---
LR		| Accuracy	| F1-Score	| Precision	| Recall	| Loss
------------------------------------------------------------------------------------------------
LR_1e-05	| 0.5254		| 0.5292		| 0.5685		| 0.4949		| 1.6146
LR_5e-05	| 0.5609		| 0.5459		| 0.5838		| 0.5127		| 1.4974
LR_0.0001	| 0.5228		| 0.5115		| 0.5478		| 0.4797		| 1.6749

--- Conclusion ---
Tuning complete. The best learning rate was LR_5e-05 with 0.5609 accuracy.
